# Multi-turn MultiSwitch - one turn after another, each on its own adapter

**Duration:** ~15 min on CPU for sections 1-8; add ~40 min and a GPU for section 9

This is the notebook for driving a **conversation** on a MultiSwitch checkpoint: ask, generate, append, ask again - with every turn naming its own adapter. Section 4 is that loop, unrolled and with nothing hidden. Everything after it is about the one decision the loop makes silently.

That decision is what happens to *history*. A control token is **markup**, not text. The chat template writes `<|uncertainty|>` into the rendered prompt, and it exists only in the token ids that render produced. The transcript you keep is `messages` - role and content, which is text - so re-rendering it on turn 2 silently drops turn 1's control token. That turn's positions are reinterpreted as base, and the KV blocks computed for them can never be reused.

Every ordinary chat API behaves that way, and it is usually fine. `Conversation` gives you the other option: `KVHistoryPolicy.PRESERVE_MIXED_HISTORY` sends the ids already sent plus the new turn, so each region keeps routing to the adapter that produced it - and stays eligible for the prefix cache.

*Why MultiSwitch:* preserving turn 1's control token puts two control tokens in one request. `SingleSwitch` **averages** competing control tokens instead of taking the most recent, so the policy refuses any checkpoint that is not `switch_type="multi"`.

**What you'll learn:**
- The four-call loop that drives a multi-turn, multi-adapter dialogue, and which call commits what
- How to run the same three turns under both policies and read per-turn routing off the real engine
- Why re-rendering a transcript is lossy, and exactly where the two prompts diverge
- How preserved control tokens become prefix-cache reuse, and why the gain is block-quantized
- The four ways to silently degrade `PRESERVE_MIXED_HISTORY` back to `RE_PREFILL`, and which of them raise

**Adapters used:** the aLoRA flavours of two intrinsics from the [Core](https://huggingface.co/ibm-granite/granitelib-core-r1.0) library (`requirement-check`, `uncertainty`).

## Prerequisites

1. **Install dependencies.** Sections 1-8 are CPU-only and need no model weights:


In [ ]:
%pip install "granite-switch[hf,compose]"

2. **Hugging Face login** - the base tokenizer and the adapter library are on the Hub:

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

3. **Disk and bandwidth.** Sections 1-8 download a *tokenizer* (a few MB) and one adapter library, and no base-model weights at all. Section 9 composes a real checkpoint: budget ~20 GB.
4. **GPU - section 9 only.** Sections 1-8 run on any laptop. Section 9 composes and serves a ~3B checkpoint and installs the `vllm` extra where it starts.

New to MultiSwitch? [`hello_multiswitch.ipynb`](./hello_multiswitch.ipynb) introduces control tokens and latest-wins routing, measures why the default engine is not enough, and closes with this same loop under the default policy. Full environment details are in [`../PREREQUISITES.md`](../PREREQUISITES.md).

---

## What re-rendering loses

Turn 1 is rendered with `adapter_name="requirement-check"`, so the template emits that adapter's control token into the ids. What you keep afterwards is `messages`, and text carries no control tokens. Render turn 2 from it and turn 1's token is gone:

```
turn 1, as sent
  <|start_of_role|>user ... spec ... <|requirement-check|> <requirements> ...
  |____________ base _______________||_______ requirement-check ______________
                                     ^ this token exists in the IDS, not in the text

turn 2, re-rendered from `messages`            (RE_PREFILL)
  <|start_of_role|>user ... spec ... <requirements> ... <|uncertainty|> ...
  |______________________ base ______________________||____ uncertainty ______
                                     ^ turn 1's token is gone. Its region now reads
                                       as base, and the blocks cached for it no
                                       longer match this prefix.

turn 2, appended to the ids already sent       (PRESERVE_MIXED_HISTORY)
  <........ exactly the ids sent in turn 1 ........> <|uncertainty|> ...
  |__ base __||___ requirement-check ___|            |____ uncertainty ______
                                     ^ unchanged token for token, so the prefix
                                       cache can still serve all of it.
```

Two separate consequences: the **routing** of history changes, and its **cached KV** stops being reusable. Sections 5 and 6 measure the routing; section 7 the reuse; section 9 confirms the reuse on a real server.

## 1 · Imports and configuration

`Conversation` is backend-agnostic: it produces token ids. Everything imported from `granite_switch.composer` is used in section 2 to build an adapter-aware tokenizer *without* downloading the base model, and everything from `granite_switch.hf` builds a switch engine on a synthetic geometry so routing can be read on CPU.

In [ ]:
import contextlib
import io
import json
import urllib.request

import torch
from transformers import AutoTokenizer

from granite_switch import Conversation, KVHistoryPolicy
from granite_switch.composer.adapter_discovery import (
    discover_adapters,
    resolve_repo_path,
)
from granite_switch.composer.arch import resolve_arch
from granite_switch.composer.tokenizer_setup import (
    add_control_tokens,
    configure_chat_template,
)
from granite_switch.config import GraniteSwitchConfig
from granite_switch.conversation import MAX_RETAINED_CONTROL_TOKENS
from granite_switch.hf.switch import create_switch

BASE_MODEL = "ibm-granite/granite-4.1-3b"
# Adapter libraries lay out adapters as <adapter>/<model>/<technology>/, and the
# directory name has no org prefix - so discovery matches on the bare model name.
TARGET_MODEL = "granite-4.1-3b"
CORE_LIB = "ibm-granite/granitelib-core-r1.0"

# Both are aLoRA. PRESERVE_MIXED_HISTORY requires that, and refuses LoRA adapters:
# a LoRA control token is emitted at sequence position 0, which is inside the
# already-sent prefix on every turn after the first. Section 8 shows the refusal.
CHECK, UNCERTAIN = "requirement-check", "uncertainty"

BLOCK = 16  # vLLM's default KV block size; reuse is only ever counted in whole blocks

print(f"base tokenizer: {BASE_MODEL}")
print(f"adapters:       {CHECK}, {UNCERTAIN}  (aLoRA, from {CORE_LIB})")

## 2 · An adapter-aware tokenizer and a switch, without the weights

In production you would use the tokenizer of a **composed** checkpoint - section 9 builds one. But the two things this notebook studies, prompt ids and routing, do not depend on a single adapter weight. So build the same tokenizer directly from the composer's own functions and skip the 6 GB download:

| Function | What it contributes |
|----------|---------------------|
| `discover_adapters` | Finds `requirement-check` and `uncertainty` in the library, and reads which technology each one is. |
| `add_control_tokens` | Adds one `<\|adapter_name\|>` special token per adapter and returns their ids. |
| `configure_chat_template` | Rewrites the Jinja template so `apply_chat_template(..., adapter_name=...)` places that adapter's token. |

Watch the `configure_chat_template` output: it reports *where* each aLoRA token will land. That placement is what decides whether `PRESERVE_MIXED_HISTORY` can work at all.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

discovered = discover_adapters(
    resolve_repo_path(CORE_LIB),
    TARGET_MODEL,
    resolve_arch(BASE_MODEL),
    technology_filter="alora",
    source=CORE_LIB,
)
discovered = [entry for entry in discovered if entry[1] in (CHECK, UNCERTAIN)]
assert {entry[1] for entry in discovered} == {CHECK, UNCERTAIN}, (
    f"expected both {CHECK!r} and {UNCERTAIN!r} as aLoRA for {TARGET_MODEL}, "
    f"found {[entry[1] for entry in discovered]}"
)

# Left in discovery order rather than sorted, because the composer does not sort
# either - so a checkpoint composed from the same library assigns the same control
# token to the same adapter, which section 9 checks.
ADAPTER_NAMES = [entry[1] for entry in discovered]

CONTROL_IDS, _special_tokens = add_control_tokens(tokenizer, discovered)
configure_chat_template(tokenizer, discovered)

`CONTROL_IDS[k]` is the token that fires adapter `k + 1`. Index `0` is always base, which is the off-by-one to keep in mind when reading routing output.

In [ ]:
INDEX_TO_NAME = {0: "base", **{k + 1: n for k, n in enumerate(ADAPTER_NAMES)}}
CONTROL_ID_SET = set(CONTROL_IDS)

print(f"{'adapter':<20} {'control token':<24} {'id':>7}  fires index")
for k, (name, token_id) in enumerate(zip(ADAPTER_NAMES, CONTROL_IDS)):
    token = f"<|{name}|>"
    assert tokenizer.convert_tokens_to_ids(token) == token_id
    print(f"{name:<20} {token:<24} {token_id:>7}  {k + 1}")

The engine itself needs no adapter weights either - only the control-token ids and the coded-memory parameters - so a `GraniteSwitchConfig` with a toy backbone geometry is enough to read routing on CPU in about a second. Same trick as section 1 of [`hello_multiswitch.ipynb`](./hello_multiswitch.ipynb), which builds both engines this way to compare them.

In [ ]:
switch_config = GraniteSwitchConfig(
    num_adapters=len(ADAPTER_NAMES),
    switch_type="multi",
    adapter_token_ids=CONTROL_IDS,
    adapter_substitute_token_ids=[1, 2],
    adapter_names=ADAPTER_NAMES,
    adapter_ranks=[8] * len(ADAPTER_NAMES),
    # Must cover the two control-token ids just added, or the token-exchange
    # lookup table is indexed past its end.
    vocab_size=len(tokenizer),
    hidden_size=256,
    num_attention_heads=4,
    num_key_value_heads=2,
    num_hidden_layers=2,
)
# from_pretrained normally sets this; a hand-built config has to.
switch_config._attn_implementation = "sdpa"
switch = create_switch(switch_config, layer_idx=0)

# This same object is what Conversation's guards read: switch_type and
# adapter_token_ids are the only two fields they need.
print(f"engine: {type(switch).__name__}  ({switch_config.num_adapters} adapters)")

## 3 · The conversation, and the helpers that read it

Three turns, each on a different adapter, in the shape a real caller would write. Note where the aLoRA **invocation text** sits: `<requirements>` and `<certainty>` go in *the turn that activates them*. That is not decoration - the template places the control token immediately before the invocation text in the **newest** user message containing it, and `PRESERVE_MIXED_HISTORY` only works when this turn's token lands in this turn's region. Section 8 demonstrates what happens when it does not.

In [ ]:
SPEC = (
    "The ingest service must accept batches of up to 5000 records, reject any batch "
    "whose checksum fails, retry transient storage errors three times with "
    "exponential backoff, and emit one audit event per accepted batch. "
)

TURNS = [
    (
        SPEC + "Our design drops the audit event when the retry budget is exhausted. "
        "Does the design meet the spec? <requirements>",
        CHECK,
        "Requirement 4 is unmet: no audit event is emitted on the exhausted-retry path.",
    ),
    (
        "How sure are you about that verdict? <certainty>",
        UNCERTAIN,
        "Moderately confident; the retry path is described in prose, not shown in code.",
    ),
    (
        "We now emit an audit event on the failure path too. Re-check it. <requirements>",
        CHECK,
        "All four requirements are met.",
    ),
]

print(f"{len(TURNS)} turns: {' -> '.join(adapter for _q, adapter, _a in TURNS)}")

Two helpers do all the reading. `route` asks the real engine which adapter each position belongs to; `spans` collapses that into contiguous runs, because a 160-position table is unreadable while four runs are not.

In [ ]:
def route(prompt_ids: list[int]) -> list[int]:
    """Per-position adapter index, from the real MultiSwitch engine."""
    indices, _modified_ids = switch.forward(
        input_ids=torch.tensor([prompt_ids]),
        adapter_token_ids=torch.tensor(CONTROL_IDS),
    )
    return indices[0].tolist()


def spans(indices: list[int]) -> list[tuple[int, int, int]]:
    """Collapse per-position indices into (start, end_inclusive, adapter_index) runs."""
    runs = []
    for position, index in enumerate(indices):
        if runs and runs[-1][2] == index:
            runs[-1][1] = position
        else:
            runs.append([position, position, index])
    return [tuple(run) for run in runs]


def control_positions(prompt_ids: list[int]) -> list[int]:
    return [i for i, token_id in enumerate(prompt_ids) if token_id in CONTROL_ID_SET]


def shared_prefix(previous: list[int], current: list[int]) -> int:
    """Length of the identical leading run - what a prefix cache can even consider."""
    count = 0
    for a, b in zip(previous, current):
        if a != b:
            break
        count += 1
    return count


def run_conversation(policy: KVHistoryPolicy) -> list[dict]:
    """Drive all three turns under one policy and record what was sent each time."""
    conversation = Conversation(tokenizer, policy=policy, config=switch_config)
    records, previous = [], []
    for turn, (question, adapter, answer) in enumerate(TURNS, start=1):
        conversation.user(question)
        prompt = list(conversation.build_prompt(adapter=adapter))
        records.append(
            {
                "turn": turn,
                "adapter": adapter,
                "prompt": prompt,
                "controls": control_positions(prompt),
                "shared": shared_prefix(previous, prompt),
                "routing": route(prompt),
            }
        )
        # Recording the answer is what commits the turn. Real ids are preferred over
        # text when the server can report them - see section 9.
        conversation.record_answer(answer, adapter=adapter)
        previous = prompt
    return records


def show(records: list[dict]) -> None:
    for record in records:
        print(
            f"turn {record['turn']}  adapter={record['adapter']:<18} "
            f"{len(record['prompt']):>4} tokens   control tokens at {record['controls']}"
        )
        for start, end, index in spans(record["routing"]):
            print(
                f"       {start:>4} - {end:<4} ({end - start + 1:>3} pos)  {INDEX_TO_NAME[index]}"
            )

## 4 · One turn after another

`run_conversation` above is a loop, which makes it easy to skim past. Unrolled, it is four calls per turn and nothing else:

```
conversation.user(question)                            # append the user turn
prompt = conversation.build_prompt(adapter=name)       # the token ids to send
answer = <send prompt, get the reply>                  # HTTP, or model.generate
conversation.record_answer(answer_ids, adapter=name)   # commit the turn
```

Two of the four are load-bearing in a way their names understate:

- **`build_prompt` does not mutate the transcript.** A prompt whose answer is never recorded - a guardian screen or judge call you discarded - leaves the conversation untouched, and the next turn's delta covers it.
- **`record_answer` is what commits the turn.** Hand it the **ids** the model emitted rather than the decoded text wherever you can: `encode(decode(ids))` is not guaranteed to reproduce `ids`, and under `PRESERVE_MIXED_HISTORY` one wrong id breaks the prefix from that point on. Section 9 takes ids off a real server; the cells below pass text, which is why they carry placeholder answers.

Here are the first two turns, run one after the other.


In [ ]:
# Placeholder answers keep sections 1-8 deterministic and weight-free. A real
# caller substitutes the model's reply here - section 9 does exactly that.
dialogue = Conversation(
    tokenizer, policy=KVHistoryPolicy.PRESERVE_MIXED_HISTORY, config=switch_config
)

question, adapter, answer = TURNS[0]
dialogue.user(question)
turn1_prompt = list(dialogue.build_prompt(adapter=adapter))
dialogue.record_answer(answer, adapter=adapter)

print(f"turn 1  adapter={adapter}")
print(
    f"        {len(turn1_prompt)} tokens, control tokens at "
    f"{control_positions(turn1_prompt)}"
)
for start, end, index in spans(route(turn1_prompt)):
    print(
        f"        {start:>4} - {end:<4} ({end - start + 1:>3} pos)  "
        f"{INDEX_TO_NAME[index]}"
    )

In [ ]:
question, adapter, answer = TURNS[1]
dialogue.user(question)
turn2_prompt = list(dialogue.build_prompt(adapter=adapter))
dialogue.record_answer(answer, adapter=adapter)

print(f"turn 2  adapter={adapter}")
print(
    f"        {len(turn2_prompt)} tokens, control tokens at "
    f"{control_positions(turn2_prompt)}"
)
for start, end, index in spans(route(turn2_prompt)):
    print(
        f"        {start:>4} - {end:<4} ({end - start + 1:>3} pos)  "
        f"{INDEX_TO_NAME[index]}"
    )
print(
    f"        reproduces turn 1's prompt for its first "
    f"{shared_prefix(turn1_prompt, turn2_prompt)} of {len(turn1_prompt)} ids"
)

# Turn 2 is the first turn that can show the policy doing anything, so assert the
# two properties it exists for rather than leaving them to the eye.
assert len(control_positions(turn2_prompt)) == 2, (
    f"turn 2 should carry both turns' control tokens, got "
    f"{control_positions(turn2_prompt)} - the policy has degraded to RE_PREFILL"
)
assert shared_prefix(turn1_prompt, turn2_prompt) == len(turn1_prompt), (
    "turn 2 must reproduce turn 1's prompt in full, or the prefix cache cannot "
    "serve any of it"
)
print("\nturn 2 kept the ids from turn 1 and added its own control token")

Turn 2 carries **two** control tokens, and its first region still routes to `requirement-check` - the adapter that produced it - while the new region routes to `uncertainty`. That is the whole point of the policy, visible on the second turn.

The rest of the notebook is the same loop with the interesting parts measured: sections 5 and 6 run all three turns under each policy, section 7 prices the difference, section 8 covers the ways to lose it, and section 9 confirms the cache reuse on a real server.


## 5 · Three turns under `RE_PREFILL` (the default)

`RE_PREFILL` re-renders the whole conversation from `messages` every turn. It needs no `config` and works with any switch type or adapter technology - it is exactly what a plain `apply_chat_template` loop already does.

In [ ]:
reprefill = run_conversation(KVHistoryPolicy.RE_PREFILL)
show(reprefill)

Every turn carries **one** control token, always near the end, and everything before it is `base`. The history is still there as text - the model can read it - but as far as the switch is concerned, turns 1 and 2 were produced by the base weights.

That is the honest description of what a re-rendered transcript is. It is only a problem when you wanted the earlier turns' regions to stay attributed to the adapters that produced them.

## 6 · The same three turns under `PRESERVE_MIXED_HISTORY`

Two things change at construction: the policy, and the now-mandatory `config`. Both of the policy's guards read it - the `switch_type` check and the control-token budget - so omitting it would not merely lose diagnostics, it would disable them. Nothing else about the calling code moves.

In [ ]:
preserve = run_conversation(KVHistoryPolicy.PRESERVE_MIXED_HISTORY)
show(preserve)

By turn 3 there are **three** control tokens in one request, one per turn, and each turn's region routes to the adapter that produced it. `requirement-check` appears twice, non-contiguously, which is precisely the write pattern `SingleSwitch` cannot represent and MultiSwitch can.

The contract is worth asserting rather than eyeballing: every position routes to the most recent control token at or before it.

In [ ]:
def latest_wins(prompt_ids: list[int]) -> list[int]:
    expected, current = [], 0
    for token_id in prompt_ids:
        if token_id in CONTROL_ID_SET:
            current = CONTROL_IDS.index(token_id) + 1
        expected.append(current)
    return expected


for policy_name, records in (("RE_PREFILL", reprefill), ("PRESERVE", preserve)):
    for record in records:
        assert record["routing"] == latest_wins(record["prompt"]), (
            f"{policy_name} turn {record['turn']}: routing does not match latest-wins"
        )
print("latest-wins routing confirmed for all 3 turns under both policies")

turn3_preserve = {INDEX_TO_NAME[i] for i in preserve[-1]["routing"]}
turn3_reprefill = {INDEX_TO_NAME[i] for i in reprefill[-1]["routing"]}
print(f"\nturn 3 adapters present   PRESERVE {sorted(turn3_preserve)}")
print(f"turn 3 adapters present   RE_PREFILL {sorted(turn3_reprefill)}")

## 7 · What it costs and what it buys

Routing is only half the reason the policy exists. The other half is that the preserved prefix is byte-identical to what was already sent, so a prefix cache can serve it.

Two numbers matter, and conflating them overstates the gain:

- **identical prefix** - how many leading ids match the previous turn's prompt. This is what changes.
- **block-aligned reuse** - `floor(identical / 16) * 16`. This is what a cache can actually hand back, because vLLM matches whole 16-token blocks.

In [ ]:
print(
    f"{'turn':<5} {'policy':<11} {'prompt':>7} {'identical':>10} {'reuse':>7} {'recomputed':>11}"
)
for turn_index in range(len(TURNS)):
    for policy_name, records in (("RE_PREFILL", reprefill), ("PRESERVE", preserve)):
        record = records[turn_index]
        reuse = record["shared"] // BLOCK * BLOCK
        print(
            f"{record['turn']:<5} {policy_name:<11} {len(record['prompt']):>7} "
            f"{record['shared']:>10} {reuse:>7} {len(record['prompt']) - reuse:>11}"
        )
    print()

Read the two arms turn by turn, because the story is not uniform:

- **Turn 2.** PRESERVE's identical prefix is longer - all of turn 1's prompt, versus RE_PREFILL diverging at turn 1's old control-token position. But both round down to the same block, so the reuse is identical here. The gain is real and simply has not crossed a boundary yet.
- **Turn 3.** The gap has grown past a block, and PRESERVE reuses one more 16-token block than RE_PREFILL.

This is the honest shape of the benefit: it grows with the conversation and it is quantized. A two-turn microbenchmark can show zero difference while the mechanism is working perfectly - which is exactly why the repo's own measurement asserts a *predicted block count* rather than a direction (`tests/integration/test_conversation_prefix_cache.py`).

The turn-2 tie above is also an artefact of *these* answers. `TURNS` carries one-line placeholder replies so the notebook stays deterministic; a real model generates more, which pushes the transcript across further block boundaries. Section 9 runs the same three turns against a live server with real generated answers, and turn 2 does separate there - 96 cache hits under `PRESERVE` against 64 under `RE_PREFILL`, two whole blocks rather than none. So read the table above as the arithmetic, not as the size of the win.

The divergence point is not arbitrary either, and it is one token earlier than you would guess. Turn 1's control token sits at position 65; the re-render of turn 2 stops matching at **64**. Print the window and the reason is visible:

```
turn 1, as sent      ... '?'   'Ġ'    '<|requirement-check|>'  'requirements'  '>'
turn 2, RE_PREFILL   ... '?'   'Ġ<'   'requirements'           '>'
                            ^^^^^^ diverges here, one token BEFORE the dropped token
```

Inserting the special token forced `Ġ<` to split into `Ġ` plus the token. Take the token away and the merge comes back, so the id at position 64 changes too. This is the same hazard `Conversation` guards against on the other side, by requiring an appended turn to begin on a special token: `tok(X + Y)` is not generally `tok(X) + tok(Y)`, and a merge spanning a join renumbers everything after it.

So the honest bound is that `RE_PREFILL` diverges *at or before* the dropped control token, while `PRESERVE` reproduces the previous prompt in full:

In [ ]:
old_control = reprefill[0]["controls"][0]
diverge = reprefill[1]["shared"]

window = range(diverge - 4, old_control + 3)
for label, prompt_ids in (
    ("turn 1, as sent", reprefill[0]["prompt"]),
    ("turn 2, RE_PREFILL", reprefill[1]["prompt"]),
):
    tokens = [tokenizer.convert_ids_to_tokens(prompt_ids[i]) for i in window]
    print(f"{label:<20}" + " ".join(f"{t!r}" for t in tokens))

print(f"\nturn 1's control token position     {old_control}")
print(f"RE_PREFILL turn 2 identical prefix  {diverge}")
print(
    f"PRESERVE   turn 2 identical prefix  {preserve[1]['shared']}"
    f"  (= turn 1's full prompt, {len(preserve[0]['prompt'])} tokens)"
)

assert diverge <= old_control, (
    f"RE_PREFILL diverged at {diverge}, after the dropped control token at "
    f"{old_control} - it cannot reproduce ids past the one it omits"
)
assert preserve[1]["shared"] == len(preserve[0]["prompt"]), (
    "PRESERVE turn 2 must reproduce turn 1's prompt in full, or the cache cannot "
    "serve any of it"
)
print("\nboth divergence points are where the mechanism predicts")

## 8 · Transport, and the four ways to get it wrong

Under `PRESERVE_MIXED_HISTORY` the prompt is only correct **as ids**. Anything that re-renders or re-tokenizes it server-side reproduces a different prefix and quietly turns the request back into `RE_PREFILL` - no error, no symptom except a fallen cache hit rate. `PromptTokenIds.requires_token_ids` exists so a transport layer can check instead of assuming.

Four failure modes, and what each one does:

| Mistake | Behaviour |
|---------|-----------|
| Posting to `/v1/chat/completions` | `chat_payload()` raises rather than return a body that would silently degrade. |
| A `switch_type="single"` checkpoint | Constructor raises: SingleSwitch averages competing control tokens, so routing would be *wrong*, not merely different. |
| Omitting `config` | Constructor raises: both guards read it, so without it they are disabled rather than merely quiet. |
| A LoRA adapter | `build_prompt` raises on turn 1, where it would otherwise render fine. |

The last one is the least obvious. A LoRA adapter's control token goes to sequence position 0, which is inside the already-sent prefix on every turn after the first - so turn 2 could never work. Turn 1 is refused anyway, because a conversation does not change adapter technology mid-dialogue: a LoRA turn 1 is a LoRA turn 2. Succeeding once and failing forever after would leave you holding a transcript that cannot continue under the policy you chose.

In [ ]:
payload_conversation = Conversation(
    tokenizer, policy=KVHistoryPolicy.PRESERVE_MIXED_HISTORY, config=switch_config
)
payload_conversation.user(TURNS[0][0])

body = payload_conversation.completion_payload(
    adapter=CHECK, max_tokens=64, temperature=0.0
)
prompt = payload_conversation.build_prompt(adapter=CHECK)

print(f"completion_payload keys : {sorted(body)}")
print(f"prompt is ids, length   : {len(body['prompt'])}")
print(f"requires_token_ids      : {prompt.requires_token_ids}")

Now each guard, fired for real. The messages are long by design - they name the mechanism and the way out, because every one of these failures is otherwise silent.

In [ ]:
def show_raise(label, thunk):
    try:
        thunk()
    except (ValueError, RuntimeError) as error:
        first_sentence = str(error).split(". ")[0]
        print(f"{label}\n    {first_sentence}.\n")
    else:
        print(f"{label}\n    NO RAISE - this guard is not working\n")


class SingleSwitchConfig:
    adapter_token_ids = CONTROL_IDS
    switch_type = "single"


show_raise(
    "chat endpoint under PRESERVE:",
    lambda: payload_conversation.chat_payload(adapter=CHECK),
)
show_raise(
    "switch_type='single':",
    lambda: Conversation(
        tokenizer,
        policy=KVHistoryPolicy.PRESERVE_MIXED_HISTORY,
        config=SingleSwitchConfig(),
    ),
)
show_raise(
    "config omitted:",
    lambda: Conversation(tokenizer, policy=KVHistoryPolicy.PRESERVE_MIXED_HISTORY),
)

The LoRA refusal needs a LoRA adapter in the template, so build a second tokenizer with the LoRA flavour of the same intrinsic. Nothing else changes - same library, same adapter name, same conversation.

In [ ]:
# The composer narrates every step; section 2 already showed that output, so keep
# this probe quiet and let the raise below be the only thing it prints.
with contextlib.redirect_stdout(io.StringIO()):
    lora_discovered = discover_adapters(
        resolve_repo_path(CORE_LIB),
        TARGET_MODEL,
        resolve_arch(BASE_MODEL),
        technology_filter="lora",
        source=CORE_LIB,
    )
    lora_discovered = [entry for entry in lora_discovered if entry[1] == CHECK]
    lora_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if lora_discovered:
        lora_ids, _ = add_control_tokens(lora_tokenizer, lora_discovered)
        configure_chat_template(lora_tokenizer, lora_discovered)

assert lora_discovered, f"no LoRA flavour of {CHECK!r} found in {CORE_LIB}"


class LoraConfig:
    adapter_token_ids = lora_ids
    switch_type = "multi"


lora_conversation = Conversation(
    lora_tokenizer, policy=KVHistoryPolicy.PRESERVE_MIXED_HISTORY, config=LoraConfig()
)
lora_conversation.user(TURNS[0][0])
show_raise(
    f"LoRA adapter {CHECK!r} on turn 1:",
    lambda: lora_conversation.build_prompt(adapter=CHECK),
)

One more failure mode has no guard, deliberately, because it has no signature to test for: an aLoRA adapter whose **invocation text sits in an older user message**. Pass 1 of the template targets the newest user message containing that text, so if the newest turn omits it the token is placed back in history - rewriting the prefix. `build_prompt` diagnoses it on the turn it happens, naming both character positions.

In [ ]:
stale = Conversation(
    tokenizer, policy=KVHistoryPolicy.PRESERVE_MIXED_HISTORY, config=switch_config
)
stale.user(TURNS[0][0])  # contains "<requirements>"
stale.build_prompt(adapter=CHECK)
stale.record_answer(TURNS[0][2], adapter=CHECK)
stale.user("And now?")  # this turn does NOT contain it

show_raise(
    "invocation text left behind in an older turn:",
    lambda: stale.build_prompt(adapter=CHECK),
)
print("The fix is in the message, not the policy: put the invocation text in the")
print("turn that activates the adapter, as TURNS does.")

Finally, two things to watch on a long-running conversation:

- **`MAX_RETAINED_CONTROL_TOKENS`** (188). The coded switch recovers a control token's write address from a `1/(1+n)` attention signal, which inverts exactly in bf16 only below that count. Preserving history is what makes `n` grow, so `build_prompt` counts and raises rather than let addresses alias silently.
- **`generated_control_tokens`**. Control tokens are freely generatable, so a model can name an adapter mid-answer and re-route the rest of its own generation. Under `PRESERVE` that persists into every later turn; under `RE_PREFILL` it is dropped at the turn boundary. A non-empty entry means the two policies no longer describe the same conversation.

In [ ]:
longest = max(len(record["controls"]) for record in preserve)
print(
    f"control tokens in the longest request : {longest} of {MAX_RETAINED_CONTROL_TOKENS}"
)

# Answers were recorded as TEXT above, so this reports None - "not checkable" - rather
# than []. By the time text arrives the token has already been stripped, and
# re-encoding cannot recover it. Section 9 records real ids and gets a real answer.
conversation = Conversation(
    tokenizer, policy=KVHistoryPolicy.PRESERVE_MIXED_HISTORY, config=switch_config
)
conversation.user(TURNS[0][0])
conversation.build_prompt(adapter=CHECK)
conversation.record_answer(TURNS[0][2], adapter=CHECK)
print(
    f"generated_control_tokens (text answer): {conversation.generated_control_tokens}"
)

## 9 · Measure the reuse on a real server (GPU)

Everything so far is a claim about token ids. This section checks the claim that matters economically - that the preserved prefix is actually served from vLLM's cache - by scraping `vllm:prefix_cache_hits_total` around each request.

**This section needs a GPU and ~20 GB of disk.** Sections 1-8 stand on their own; stop here if you are on a laptop.

Three things about the setup are load-bearing:

- **`--enable-prefix-caching`** must be on. It is vLLM V1's default, but a future flip would zero the whole experiment silently.
- **`--return-tokens-as-token-ids`** plus `logprobs: 1` makes the server report each generated token as the literal string `token_id:NNNN`, so the transcript can be extended with the ids the model really emitted. Re-encoding the returned *text* is not safe: `encode(detokenize(ids))` does not always reproduce `ids`, and one wrong id makes the block straddling the prompt/answer seam miss - capping reuse at the last boundary before the answer, which is exactly the saving being measured. The repo's own test failed this way once, at 64 hits where 96 were available.
- **The two arms must not share their opening**, or whichever policy runs second finds the first one's turn-1 blocks already cached. They must also be the **same length**, or their absolute hit counts are not comparable for a reason that has nothing to do with the policy.

In [ ]:
%pip install "granite-switch[vllm]"

Compose the checkpoint. `--include-adapters` restricts it to the same two adapters section 2 used, so the control-token ids come out identical and the next cell can assert it.

In [ ]:
MODEL_OUT = "./granite-switch-conv"

!python -m granite_switch.composer.compose_granite_switch   --base-model {BASE_MODEL}   --adapters {CORE_LIB}   --include-adapters {CHECK} {UNCERTAIN}   --technology-filter alora   --switch-type multi   --output {MODEL_OUT}

Worth asserting rather than assuming. If `--switch-type` had not persisted, `from_pretrained` would build a `SingleSwitch`, `Conversation` would refuse the policy, and the failure would at least be loud - but the id check below is what confirms this checkpoint is the same one sections 2-8 reasoned about.

In [ ]:
served_config = GraniteSwitchConfig.from_pretrained(MODEL_OUT)
served_tokenizer = AutoTokenizer.from_pretrained(MODEL_OUT)

assert served_config.switch_type == "multi", (
    f"switch_type is {served_config.switch_type!r}; --switch-type did not persist"
)
# Compare the name -> token-id mapping, not the two lists: adapter order is
# discovery order, and only the per-adapter assignment has to agree.
served_assignment = dict(
    zip(served_config.adapter_names, served_config.adapter_token_ids)
)
notebook_assignment = dict(zip(ADAPTER_NAMES, CONTROL_IDS))
assert served_assignment == notebook_assignment, (
    f"composed checkpoint assigns {served_assignment}, section 2 assigned "
    f"{notebook_assignment} - this is not the same adapter set"
)
print(f"switch_type      : {served_config.switch_type}")
print(f"adapters         : {list(served_config.adapter_names)}")
print(f"control token ids: {served_assignment}  (matches section 2)")

Start the server with prefix caching on and token ids reported.

In [ ]:
from granite_switch.tutorials.vllm_server import (
    kill_stale_vllm_processes,
    launch_vllm,
    tail_log,
    wait_for_server,
)

VLLM_PORT = 8137
VLLM_LOG = "./vllm_conversation.log"

kill_stale_vllm_processes()  # a restarted notebook can leave a process holding the GPU

vllm_proc = launch_vllm(
    model=MODEL_OUT,
    port=VLLM_PORT,
    log_file=VLLM_LOG,
    max_num_seqs=4,
    max_model_len=4096,
    enforce_eager=True,  # skips CUDA-graph capture; faster startup for a tutorial
    extra_args=["--enable-prefix-caching", "--return-tokens-as-token-ids"],
)

if not wait_for_server(VLLM_PORT, log_file=VLLM_LOG):
    tail_log(VLLM_LOG, n=40)
    raise RuntimeError("vLLM did not come up - see the log tail above")

Two request helpers: one to post ids to `/v1/completions`, one to read the prefix-cache counters. The counters are cumulative, so each turn's figure is a delta measured around its own request.

In [ ]:
TOKEN_ID_PREFIX = "token_id:"


def post_completion(payload, timeout=240):
    request = urllib.request.Request(
        f"http://127.0.0.1:{VLLM_PORT}/v1/completions",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode())["choices"][0]


def prefix_cache_counters():
    counters = {}
    with urllib.request.urlopen(
        f"http://127.0.0.1:{VLLM_PORT}/metrics", timeout=20
    ) as response:
        for line in response.read().decode().splitlines():
            for key in (
                "vllm:prefix_cache_queries_total",
                "vllm:prefix_cache_hits_total",
            ):
                if line.startswith(key):
                    counters[key] = counters.get(key, 0.0) + float(
                        line.rsplit(" ", 1)[1]
                    )
    return counters


def generated_ids(choice):
    """Exact ids the model emitted, or None if this build cannot report them."""
    tokens = (choice.get("logprobs") or {}).get("tokens") or []
    ids = []
    for token in tokens:
        if not str(token).startswith(TOKEN_ID_PREFIX):
            return None
        ids.append(int(str(token)[len(TOKEN_ID_PREFIX) :]))
    return ids or None

Now run both policies against the server. `CASE_MARKER` gives each arm a disjoint opening of **equal length** - the two constraints named above, at no extra cost.

In [ ]:
CASE_MARKER = {
    KVHistoryPolicy.RE_PREFILL: "Case A. ",
    KVHistoryPolicy.PRESERVE_MIXED_HISTORY: "Case B. ",
}
MAX_TOKENS = 24

served = {}
for policy in (KVHistoryPolicy.RE_PREFILL, KVHistoryPolicy.PRESERVE_MIXED_HISTORY):
    conversation = Conversation(served_tokenizer, policy=policy, config=served_config)
    turns = []
    for turn, (question, adapter, _demo_answer) in enumerate(TURNS, start=1):
        marked = CASE_MARKER[policy] + question if turn == 1 else question
        conversation.user(marked)
        prompt = list(conversation.build_prompt(adapter=adapter))

        before = prefix_cache_counters()
        choice = post_completion(
            {
                "model": MODEL_OUT,
                "prompt": prompt,
                "max_tokens": MAX_TOKENS,
                "temperature": 0.0,
                "logprobs": 1,  # required for the server to report tokens at all
            }
        )
        after = prefix_cache_counters()

        answer_ids = generated_ids(choice)
        conversation.record_answer(answer_ids or choice["text"], adapter=adapter)
        turns.append(
            {
                "turn": turn,
                "adapter": adapter,
                "prompt_len": len(prompt),
                "controls": [i for i, t in enumerate(prompt) if t in CONTROL_ID_SET],
                "hits": after["vllm:prefix_cache_hits_total"]
                - before["vllm:prefix_cache_hits_total"],
                "queries": after["vllm:prefix_cache_queries_total"]
                - before["vllm:prefix_cache_queries_total"],
                "ids_exact": answer_ids is not None,
                # The cache can only match what was literally sent.
                "transcript_extends_prompt": conversation.sent_token_ids[: len(prompt)]
                == prompt,
            }
        )
    served[policy] = turns

print(f"{'turn':<5} {'policy':<11} {'prompt':>7} {'hits':>6} {'queries':>8}  controls")
for policy, turns in served.items():
    for record in turns:
        label = "RE_PREFILL" if policy is KVHistoryPolicy.RE_PREFILL else "PRESERVE"
        print(
            f"{record['turn']:<5} {label:<11} {record['prompt_len']:>7} "
            f"{record['hits']:>6.0f} {record['queries']:>8.0f}  {record['controls']}"
        )

Check the measurement before believing it. Each assertion below corresponds to a specific way this could report a false pass: caching off, ids re-encoded from text, a transcript that does not begin with what was sent, or a PRESERVE prompt that never carried the extra control token.

In [ ]:
reprefill_turns = served[KVHistoryPolicy.RE_PREFILL]
preserve_turns = served[KVHistoryPolicy.PRESERVE_MIXED_HISTORY]

for label, turns in (("RE_PREFILL", reprefill_turns), ("PRESERVE", preserve_turns)):
    assert turns[-1]["queries"] > 0, (
        f"{label}: vLLM reported zero prefix-cache queries, so caching was not on "
        "and this comparison measures nothing"
    )
    assert turns[-1]["ids_exact"], (
        f"{label}: the answer was re-encoded from text. One wrong id makes the block "
        "straddling the prompt/answer seam miss, capping reuse at the last boundary "
        "before the answer - the very saving being measured. Needs a vLLM build that "
        "supports --return-tokens-as-token-ids."
    )
assert all(record["transcript_extends_prompt"] for record in preserve_turns), (
    "the PRESERVE transcript does not begin with what was actually sent, so no reuse "
    "is possible by construction"
)
assert len(preserve_turns[-1]["controls"]) == len(TURNS), (
    f"PRESERVE turn {len(TURNS)} should carry one control token per turn, got "
    f"{preserve_turns[-1]['controls']}"
)
assert len(reprefill_turns[-1]["controls"]) == 1, (
    f"RE_PREFILL should carry only the current turn's control token, got "
    f"{reprefill_turns[-1]['controls']}"
)

final_preserve = preserve_turns[-1]["hits"]
final_reprefill = reprefill_turns[-1]["hits"]
print(
    f"final-turn prefix-cache hits   PRESERVE {final_preserve:.0f}   RE_PREFILL {final_reprefill:.0f}"
)
assert final_preserve > final_reprefill, (
    f"PRESERVE reused no more than RE_PREFILL ({final_preserve:.0f} vs "
    f"{final_reprefill:.0f}); keeping the ids cost a control token and bought nothing"
)
print("PRESERVE reused strictly more of the final turn's prompt")

For reference, the same experiment measured on 1x A100 (vLLM 0.19.x, `granite-4.1-3b` with two RAG aLoRA adapters) in `tests/integration/test_conversation_prefix_cache.py`, on a two-turn version of this conversation:

| policy | turn-2 prompt | positions recomputed | prefix-cache hits |
|--------|---------------|----------------------|-------------------|
| `RE_PREFILL` | 102 | 38 | 64 |
| `PRESERVE_MIXED_HISTORY` | 104 | 24 | 80 |

Both hit counts land exactly where the block arithmetic of section 7 predicts: `RE_PREFILL` diverges at its dropped control token (index 77, so `floor(77/16) * 16 = 64`) and `PRESERVE` at its transcript boundary (89, so `80`). Your numbers will differ with adapter set and message lengths; the arithmetic is what should hold.

Shut the server down when you are done - it holds the GPU.

In [ ]:
vllm_proc.terminate()
vllm_proc.wait(timeout=60)
print("server stopped")

## 10 · Next steps

- **Adapt it.** Section 4 is the whole pattern: `user()`, `build_prompt(adapter=...)`, send the ids, `record_answer(ids)`. Swap `TURNS` for your own dialogue and keep the invocation text in the turn that activates each adapter.
- **Start further back if the routing itself is new.** [`hello_multiswitch.ipynb`](./hello_multiswitch.ipynb) measures why `single` cannot do this - it mis-routes 54 of 64 three-hop orderings - then composes a `multi` checkpoint and reads the per-token decision off the model.
- **Serve it under load.** [`multiswitch_serving.ipynb`](./multiswitch_serving.ipynb) covers batching and cross-request isolation on the same kind of server section 9 starts.
- **Compose for a different adapter set.** [`compose_granite_switch.ipynb`](./compose_granite_switch.ipynb) is the full tour of the composer's selection flags, including mixing LoRA and aLoRA - which changes where control tokens land and therefore which policy applies.
- **Read the mechanism end to end.** Section 6 of [`../../docs/MULTISWITCH_EXPLAINED.html`](../../docs/MULTISWITCH_EXPLAINED.html) walks the cross-turn KV story and the Conversation API in detail.
